In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 96.9 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import json
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
import torch.nn.functional as F
from tqdm import tqdm
from gensim.models import FastText
import os
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import re

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [6]:
model_13th = FastText.load("/content/drive/MyDrive/LSD-Analysis/Fasttext/models/sinhala_ft_century_13.model")
model_14th = FastText.load("/content/drive/MyDrive/LSD-Analysis/Fasttext/models/sinhala_ft_century_14.model")
model_15th = FastText.load("/content/drive/MyDrive/LSD-Analysis/Fasttext/models/sinhala_ft_century_15.model")
model_18th = FastText.load("/content/drive/MyDrive/LSD-Analysis/Fasttext/models/sinhala_ft_century_18.model")
model_19th = FastText.load("/content/drive/MyDrive/LSD-Analysis/Fasttext/models/sinhala_ft_century_19.model")
model_20th = FastText.load("/content/drive/MyDrive/LSD-Analysis/Fasttext/models/sinhala_ft_century_20.model")

In [7]:
with open("/content/drive/MyDrive/LSD-Analysis/Data/consistent_keys_2.json", "r", encoding="utf-8") as f:
    consistent_keys = json.load(f)

In [8]:
len(consistent_keys)

210

In [9]:
century_models = {
    # '5th': model_5th,
    # '12th': model_12th,
    '13th': model_13th,
    '14th': model_14th,
    '15th': model_15th,
    # '16th': model_16th,
    '18th': model_18th,
    '19th': model_19th,
    '20th': model_20th
    }

In [10]:
century_names = sorted(list(century_models.keys()))

In [11]:
learned_weights_matrices = {}

In [12]:
# Hyperparameters for training
embedding_size = 300
learning_rate = 0.001
epochs = 2000
lambda_ortho = 0.001 # Penalty to keep W rotational (orthogonal)

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [14]:
# --- Configuration ---
anchor_name = "13th"
patience_limit = 20
patience_threshold = 1e-7
min_epochs = 100
report_interval = 50       # How often to print epoch updates

alignment_quality_metrics = []
unified_space = {}
learned_weights_matrices = {}

# 1. Setup the Anchor (13th Century)
raw_Y = torch.tensor([century_models[anchor_name].wv[word] for word in consistent_keys], dtype=torch.float32)
Y_ref = F.normalize(raw_Y, p=2, dim=1)
Y_ref_np = Y_ref.detach().numpy()

unified_space[anchor_name] = Y_ref_np

for target_name in century_names:
    if target_name == anchor_name:
        continue

    # 2. Prepare Source Vectors (Target Century)
    raw_X = torch.tensor([century_models[target_name].wv[word] for word in consistent_keys], dtype=torch.float32)
    X_input = F.normalize(raw_X, p=2, dim=1)

    # 3. Model Setup: Map Target -> Anchor Space
    model = nn.Linear(embedding_size, embedding_size, bias=False)
    nn.init.eye_(model.weight)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    best_loss = float('inf')
    patience_counter = 0

    print(f"\n" + "="*50)
    print(f"ALIGNING: {target_name} ---> {anchor_name}")
    print("="*50)

    # 4. Training Loop
    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X_input)

        mse_loss = criterion(predictions, Y_ref)

        # Orthogonality Constraint
        W_curr = model.weight
        ortho_loss = torch.norm(torch.mm(W_curr, W_curr.t()) - torch.eye(embedding_size), p='fro')**2

        loss = mse_loss + (lambda_ortho * ortho_loss)
        loss.backward()
        optimizer.step()

        current_loss = loss.item()

        # Print Epoch Progress
        if (epoch + 1) % report_interval == 0 or epoch == 0:
            print(f"Epoch {epoch+1:4d} | Total Loss: {current_loss:.7f} | MSE: {mse_loss.item():.7f} | Ortho: {ortho_loss.item():.7f}")

        # --- Early Stopping Logic ---
        if (best_loss - current_loss) > patience_threshold:
            best_loss = current_loss
            patience_counter = 0
        else:
            patience_counter += 1

        # Check for Convergence after min_epochs
        if patience_counter >= patience_limit and epoch >= min_epochs:
            print(f"--- Converged at epoch {epoch+1} ---")
            break

    # 5. Final Alignment and Quality Measurement
    with torch.no_grad():
        Q = model.weight.detach().numpy()
        # Transform the target space into the anchor space
        X_aligned = (X_input @ torch.tensor(Q).t()).numpy()

        unified_space[target_name] = X_aligned
        learned_weights_matrices[f"{target_name}_to_{anchor_name}"] = Q

        # --- INFUSION: MEASURE ALIGNMENT QUALITY ---
        # Procrustes Disparity (Sum of Squared Errors)
        disparity = np.sum(np.square(X_aligned - Y_ref_np))

        # Rotational Distance (Frobenius norm of difference from Identity)
        rotational_distance = np.linalg.norm(Q - np.eye(embedding_size), ord='fro')

        alignment_quality_metrics.append({
            "Target_Century": target_name,
            "Reference_Century": anchor_name,
            "Procrustes_Disparity": round(float(disparity), 6),
            "Rotational_Distance": round(float(rotational_distance), 6),
            "Epochs_Completed": epoch + 1
        })

# Final Summary Table
metrics_df = pd.DataFrame(alignment_quality_metrics)
print("\n" + "!"*30)
print("FINAL ALIGNMENT REPORT")
print("!"*30)
print(metrics_df)

/tmp/ipykernel_2344/399615730.py:13: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  raw_Y = torch.tensor([century_models[anchor_name].wv[word] for word in consistent_keys], dtype=torch.float32)



ALIGNING: 14th ---> 13th
Epoch    1 | Total Loss: 0.0032704 | MSE: 0.0032704 | Ortho: 0.0000000
Epoch   50 | Total Loss: 0.0023726 | MSE: 0.0023626 | Ortho: 0.0100540
Epoch  100 | Total Loss: 0.0019622 | MSE: 0.0019541 | Ortho: 0.0081560
Epoch  150 | Total Loss: 0.0017262 | MSE: 0.0017185 | Ortho: 0.0076660
Epoch  200 | Total Loss: 0.0015663 | MSE: 0.0015589 | Ortho: 0.0073860
Epoch  250 | Total Loss: 0.0014486 | MSE: 0.0014414 | Ortho: 0.0072116
Epoch  300 | Total Loss: 0.0013577 | MSE: 0.0013506 | Ortho: 0.0071133
Epoch  350 | Total Loss: 0.0012856 | MSE: 0.0012785 | Ortho: 0.0070341
Epoch  400 | Total Loss: 0.0012274 | MSE: 0.0012204 | Ortho: 0.0069976
Epoch  450 | Total Loss: 0.0011800 | MSE: 0.0011731 | Ortho: 0.0069640
Epoch  500 | Total Loss: 0.0011411 | MSE: 0.0011342 | Ortho: 0.0069361
Epoch  550 | Total Loss: 0.0011089 | MSE: 0.0011020 | Ortho: 0.0069223
Epoch  600 | Total Loss: 0.0010822 | MSE: 0.0010753 | Ortho: 0.0069293
Epoch  650 | Total Loss: 0.0010599 | MSE: 0.0010530

Epoch 1650 | Total Loss: 0.0009470 | MSE: 0.0009399 | Ortho: 0.0070823
--- Converged at epoch 1657 ---

ALIGNING: 15th ---> 13th
Epoch    1 | Total Loss: 0.0033125 | MSE: 0.0033125 | Ortho: 0.0000000
Epoch   50 | Total Loss: 0.0023920 | MSE: 0.0023822 | Ortho: 0.0098026
Epoch  100 | Total Loss: 0.0019637 | MSE: 0.0019559 | Ortho: 0.0078039
Epoch  150 | Total Loss: 0.0017162 | MSE: 0.0017090 | Ortho: 0.0072001
Epoch  200 | Total Loss: 0.0015487 | MSE: 0.0015418 | Ortho: 0.0068433
Epoch  250 | Total Loss: 0.0014255 | MSE: 0.0014188 | Ortho: 0.0066170
Epoch  300 | Total Loss: 0.0013305 | MSE: 0.0013240 | Ortho: 0.0064719
Epoch  350 | Total Loss: 0.0012553 | MSE: 0.0012489 | Ortho: 0.0063669
Epoch  400 | Total Loss: 0.0011947 | MSE: 0.0011884 | Ortho: 0.0063076
Epoch  450 | Total Loss: 0.0011453 | MSE: 0.0011391 | Ortho: 0.0062616
Epoch  500 | Total Loss: 0.0011048 | MSE: 0.0010985 | Ortho: 0.0062300
Epoch  550 | Total Loss: 0.0010713 | MSE: 0.0010651 | Ortho: 0.0061892
Epoch  600 | Total 

Epoch 1650 | Total Loss: 0.0009013 | MSE: 0.0008948 | Ortho: 0.0064502
Epoch 1700 | Total Loss: 0.0009006 | MSE: 0.0008942 | Ortho: 0.0064000
--- Converged at epoch 1704 ---

ALIGNING: 18th ---> 13th
Epoch    1 | Total Loss: 0.0034906 | MSE: 0.0034906 | Ortho: 0.0000000
Epoch   50 | Total Loss: 0.0025518 | MSE: 0.0025364 | Ortho: 0.0153667
Epoch  100 | Total Loss: 0.0021237 | MSE: 0.0021098 | Ortho: 0.0138592
Epoch  150 | Total Loss: 0.0018783 | MSE: 0.0018649 | Ortho: 0.0134402
Epoch  200 | Total Loss: 0.0017139 | MSE: 0.0017007 | Ortho: 0.0132080
Epoch  250 | Total Loss: 0.0015940 | MSE: 0.0015810 | Ortho: 0.0130793
Epoch  300 | Total Loss: 0.0015024 | MSE: 0.0014894 | Ortho: 0.0129996
Epoch  350 | Total Loss: 0.0014303 | MSE: 0.0014174 | Ortho: 0.0129509
Epoch  400 | Total Loss: 0.0013725 | MSE: 0.0013596 | Ortho: 0.0129328
Epoch  450 | Total Loss: 0.0013256 | MSE: 0.0013127 | Ortho: 0.0129294
Epoch  500 | Total Loss: 0.0012873 | MSE: 0.0012744 | Ortho: 0.0129272
Epoch  550 | Total 

--- Converged at epoch 1547 ---

ALIGNING: 19th ---> 13th
Epoch    1 | Total Loss: 0.0032646 | MSE: 0.0032646 | Ortho: 0.0000000
Epoch   50 | Total Loss: 0.0023351 | MSE: 0.0023267 | Ortho: 0.0083496
Epoch  100 | Total Loss: 0.0019020 | MSE: 0.0018959 | Ortho: 0.0061326
Epoch  150 | Total Loss: 0.0016535 | MSE: 0.0016481 | Ortho: 0.0054480
Epoch  200 | Total Loss: 0.0014854 | MSE: 0.0014804 | Ortho: 0.0050318
Epoch  250 | Total Loss: 0.0013615 | MSE: 0.0013567 | Ortho: 0.0047614
Epoch  300 | Total Loss: 0.0012657 | MSE: 0.0012611 | Ortho: 0.0045865
Epoch  350 | Total Loss: 0.0011895 | MSE: 0.0011851 | Ortho: 0.0044609
Epoch  400 | Total Loss: 0.0011280 | MSE: 0.0011236 | Ortho: 0.0043781
Epoch  450 | Total Loss: 0.0010776 | MSE: 0.0010733 | Ortho: 0.0043129
Epoch  500 | Total Loss: 0.0010362 | MSE: 0.0010319 | Ortho: 0.0042633
Epoch  550 | Total Loss: 0.0010019 | MSE: 0.0009976 | Ortho: 0.0042453
Epoch  600 | Total Loss: 0.0009733 | MSE: 0.0009691 | Ortho: 0.0042052
Epoch  650 | Total 

Epoch 1600 | Total Loss: 0.0008274 | MSE: 0.0008232 | Ortho: 0.0042138
--- Converged at epoch 1643 ---

ALIGNING: 20th ---> 13th
Epoch    1 | Total Loss: 0.0032457 | MSE: 0.0032457 | Ortho: 0.0000000
Epoch   50 | Total Loss: 0.0023090 | MSE: 0.0023008 | Ortho: 0.0081381
Epoch  100 | Total Loss: 0.0018683 | MSE: 0.0018625 | Ortho: 0.0058547
Epoch  150 | Total Loss: 0.0016176 | MSE: 0.0016125 | Ortho: 0.0051476
Epoch  200 | Total Loss: 0.0014486 | MSE: 0.0014439 | Ortho: 0.0047179
Epoch  250 | Total Loss: 0.0013241 | MSE: 0.0013196 | Ortho: 0.0044403
Epoch  300 | Total Loss: 0.0012278 | MSE: 0.0012235 | Ortho: 0.0042512
Epoch  350 | Total Loss: 0.0011512 | MSE: 0.0011470 | Ortho: 0.0041177
Epoch  400 | Total Loss: 0.0010892 | MSE: 0.0010852 | Ortho: 0.0040194
Epoch  450 | Total Loss: 0.0010387 | MSE: 0.0010347 | Ortho: 0.0039643
Epoch  500 | Total Loss: 0.0009971 | MSE: 0.0009932 | Ortho: 0.0039215
Epoch  550 | Total Loss: 0.0009628 | MSE: 0.0009589 | Ortho: 0.0038864
Epoch  600 | Total 

In [15]:
print(f"Learned {len(learned_weights_matrices)} transformation matrices (W).")

Learned 5 transformation matrices (W).


In [16]:
# Path for the complete set
save_path = "/content/drive/MyDrive/LSD-Analysis/Fasttext/models/sinhala_all_alignments.npz"
np.savez_compressed(save_path, **unified_space)

In [18]:
loaded_data = np.load(save_path)
unified_space = {key: loaded_data[key] for key in loaded_data.files}
print(f"Loaded centuries: {list(unified_space.keys())}")

Loaded centuries: ['13th', '14th', '15th', '18th', '19th', '20th']


In [19]:
output_dir = '/content/drive/MyDrive/LSD-Analysis/Data/Word_Similarity_Heatmaps_Method_3_FT'

In [20]:
font_path = '/content/drive/MyDrive/LSD-Analysis/Fonts/Noto_Sans_Sinhala/NotoSansSinhala-VariableFont_wdth,wght.ttf'
fm.fontManager.addfont(font_path)
sinhala_font_name = fm.FontProperties(fname=font_path).get_name()
fm._load_fontmanager(try_read_cache=False)

In [21]:
# 1. Map consistent_keys to their index for quick lookup
word_to_idx = {word: i for i, word in enumerate(consistent_keys)}
centuries = sorted(unified_space.keys()) # Ensure chronological order

for word in tqdm(consistent_keys, desc="Generating Heatmaps"):
    word_idx = word_to_idx[word]

    # 2. Build the Similarity Matrix for this specific word
    # Rows = Year A, Cols = Year B
    sim_matrix = np.zeros((len(centuries), len(centuries)))

    for i, year_a in enumerate(centuries):
        for j, year_b in enumerate(centuries):
            # Extract word vectors from the unified space
            vec_a = unified_space[year_a][word_idx].reshape(1, -1)
            vec_b = unified_space[year_b][word_idx].reshape(1, -1)

            # Calculate Similarity
            sim = cosine_similarity(vec_a, vec_b)[0][0]
            sim_matrix[i, j] = sim

    # 3. Convert to DataFrame for Seaborn
    heatmap_df = pd.DataFrame(sim_matrix, index=centuries, columns=centuries)

    # 4. Plotting
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        heatmap_df,
        annot=True,
        fmt=".3f",
        cmap="YlGnBu",
        linewidths=0.5,
        linecolor='white',
        cbar_kws={'label': 'Unified Space Cosine Similarity'}
    )

    plt.title(f'Cross-Century Semantic Stability: "{word}"', fontsize=16, fontproperties=sinhala_font_name)
    plt.xlabel('Century B', fontsize=12, fontproperties=sinhala_font_name)
    plt.ylabel('Century A', fontsize=12, fontproperties=sinhala_font_name)

    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()

    # 5. Save logic
    safe_key = str(word).replace(' ', '_').replace('/', '_')
    filename = os.path.join(output_dir, f"{safe_key}_similarity.png")

    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

print(f"\nSuccess: {len(consistent_keys)} heatmaps saved to '{output_dir}'.")

Generating Heatmaps: 100%|██████████| 210/210 [03:42<00:00,  1.06s/it]


Success: 210 heatmaps saved to '/content/drive/MyDrive/LSD-Analysis/Data/Word_Similarity_Heatmaps_Method_3_FT'.


In [22]:
centuries = sorted(unified_space.keys(), key=lambda x: int(''.join(filter(str.isdigit, x))))
word_to_idx = {word: i for i, word in enumerate(consistent_keys)}

all_rows = []

# 2. Iterate through each word
for word in consistent_keys:
    word_idx = word_to_idx[word]

    # 3. For each word, create a row for every 'Year A'
    for year_a in centuries:
        row_data = {
            'Key': word,
            'Year_A': year_a
        }

        # Get vector for Year A
        vec_a = unified_space[year_a][word_idx].reshape(1, -1)

        # 4. Compare Year A against all possible 'Year B's
        for year_b in centuries:
            vec_b = unified_space[year_b][word_idx].reshape(1, -1)
            sim = cosine_similarity(vec_a, vec_b)[0][0]

            # Use the century name as the column name (Year B)
            row_data[year_b] = round(float(sim), 6)

        all_rows.append(row_data)

# 5. Create the final DataFrame
final_df_org = pd.DataFrame(all_rows)

# 6. Sort by Key (Word) and then chronologically by Year_A
final_df_org = final_df_org.sort_values(by=['Key', 'Year_A']).reset_index(drop=True)

In [23]:
final_df_org.head(10)

,Key,Year_A,13th,14th,15th,18th,19th,20th
0,අක්ෂර,13th,1.000000,0.888121,0.917643,0.888640,0.839533,0.900111
1,අක්ෂර,14th,0.888121,1.000000,0.878215,0.874141,0.809296,0.874968
2,අක්ෂර,15th,0.917643,0.878215,1.000000,0.881346,0.860073,0.905512
3,අක්ෂර,18th,0.888640,0.874141,0.881346,1.000000,0.811627,0.863354
4,අක්ෂර,19th,0.839533,0.809296,0.860073,0.811627,1.000000,0.863152
5,අක්ෂර,20th,0.900111,0.874968,0.905512,0.863354,0.863152,1.000000
6,අග්ර,13th,1.000000,0.873194,0.862814,0.806993,0.831184,0.850666
7,අග්ර,14th,0.873194,1.000000,0.865016,0.827354,0.856143,0.863327
8,අග්ර,15th,0.862814,0.865016,1.000000,0.801348,0.817298,0.884720
9,අග්ර,18th,0.806993,0.827354,0.801348,1.000000,0.835205,0.784615


In [24]:
def extract_year(name):
    try:
        return int(''.join(filter(str.isdigit, str(name))))
    except ValueError:
        return 0

In [25]:
df_long = final_df_org.melt(id_vars=['Key', 'Year_A'], var_name='Year_B', value_name='Value')
df_long['Year_A_num'] = df_long['Year_A'].apply(extract_year)
df_long['Year_B_num'] = df_long['Year_B'].apply(extract_year)

df_long['feature_pair'] = df_long['Year_A'].astype(str) + '_' + df_long['Year_B'].astype(str)

df_lags_only = df_long[df_long['Year_A_num'] < df_long['Year_B_num']].copy()

piv_df_org = df_lags_only.pivot(index='Key', columns='feature_pair', values='Value')
piv_df_org['Mean_Similarity'] = piv_df_org.mean(axis=1, numeric_only=True)
piv_df_org = piv_df_org.reset_index()

In [26]:
piv_df_org.head()

feature_pair,Key,13th_14th,13th_15th,13th_18th,13th_19th,13th_20th,14th_15th,14th_18th,14th_19th,14th_20th,15th_18th,15th_19th,15th_20th,18th_19th,18th_20th,19th_20th,Mean_Similarity
0,අක්ෂර,0.888121,0.917643,0.888640,0.839533,0.900111,0.878215,0.874141,0.809296,0.874968,0.881346,0.860073,0.905512,0.811627,0.863354,0.863152,0.870382
1,අග්ර,0.873194,0.862814,0.806993,0.831184,0.850666,0.865016,0.827354,0.856143,0.863327,0.801348,0.817298,0.884720,0.835205,0.784615,0.842420,0.840153
2,අතුරෙන්,0.782476,0.908421,0.784053,0.887121,0.929245,0.803780,0.805067,0.821003,0.785710,0.809538,0.885222,0.906379,0.797092,0.801872,0.894805,0.840119
3,අන්ය,0.763283,0.863902,0.707404,0.781157,0.882906,0.704465,0.719479,0.642360,0.727799,0.669926,0.826045,0.824040,0.581954,0.650363,0.754198,0.739952
4,අනෙක,0.842110,0.831221,0.880982,0.894536,0.882349,0.823121,0.847510,0.831046,0.838645,0.850525,0.792859,0.789415,0.855466,0.823512,0.857766,0.842738


In [27]:
piv_df_org['Mean_Similarity'].describe()

,Mean_Similarity
count,210.000000
mean,0.810353
std,0.157162
min,0.000000
25%,0.813888
50%,0.840576
75%,0.864800
max,0.949865


In [ ]:
def get_semantic_drift_report(df, models_dict, threshold_std=0.8, top_n=10):
    results = []
    # Ensure 'Key' is the index
    work_df = df.set_index('Key') if 'Key' in df.columns else df.copy()

    # Identify all comparison columns (13_14, 13_15, 14_15, etc.)
    cols = [c for c in work_df.columns if isinstance(c, str) and '_' in c]

    for col in cols:
        parts = col.split('_')
        try:
            y_start, y_end = int(parts[0]), int(parts[1])
        except: continue

        if y_start not in models_dict or y_end not in models_dict:
            continue

        m_start = models_dict[y_start]
        m_end = models_dict[y_end]

        # Calculate a dynamic cutoff for this specific period
        series = work_df[col][work_df[col] > 0]
        if series.empty: continue

        cutoff = series.mean() - (threshold_std * series.std())
        dips = series[series < cutoff]

        for word, score in dips.items():
            try:
                # 1. Check vocab existence
                if word not in m_start.wv or word not in m_end.wv:
                    continue

                # 2. Dynamic top_n check to prevent IndexError
                # If vocab is smaller than requested top_n, adjust it
                v_start_size = len(m_start.wv.index_to_key)
                v_end_size = len(m_end.wv.index_to_key)
                safe_n = min(top_n, v_start_size - 1, v_end_size - 1)

                if safe_n < 1: continue

                # 3. Retrieve neighbors with explicit error handling for the Gensim bug
                try:
                    n_start_raw = m_start.wv.most_similar(word, topn=safe_n)
                    n_end_raw = m_end.wv.most_similar(word, topn=safe_n)
                except IndexError:
                    # If Gensim's index is still broken, force a refresh and try one last time
                    m_start.wv.fill_norms(force=True)
                    m_end.wv.fill_norms(force=True)
                    n_start_raw = m_start.wv.most_similar(word, topn=safe_n)
                    n_end_raw = m_end.wv.most_similar(word, topn=safe_n)

                n_start = [n[0] for n in n_start_raw]
                n_end = [n[0] for n in n_end_raw]

                results.append({
                    "Word": word,
                    "Period": col,
                    "Similarity": round(score, 4),
                    f"Neighbors_{y_start}": ", ".join(n_start),
                    f"Neighbors_{y_end}": ", ".join(n_end)
                })
            except Exception:
                # Catch-all for any remaining weirdness with specific words
                continue

    return pd.DataFrame(results)

In [ ]:
# --- 1. Create a temporary models dictionary with integer keys ---
# This converts {"13th_century": model} -> {13: model}
int_models_dict = {}
for key, model in century_models.items():
    # Extract just the digits (e.g., "13th_century" -> 13)
    int_key = int(''.join(filter(str.isdigit, key)))
    int_models_dict[int_key] = model

# --- 2. Create a temporary DataFrame with integer column names ---
# This converts "13th_century_14th_century" -> "13_14"
piv_df_translated = piv_df_org.copy()
new_cols = {'Key': 'Key'}

for col in piv_df_org.columns:
    if col == 'Key': continue
    # Find all numbers in the column string
    nums = re.findall(r'\d+', col)
    if len(nums) >= 2:
        new_cols[col] = f"{nums[0]}_{nums[1]}"

piv_df_translated = piv_df_translated.rename(columns=new_cols)

# --- 3. Run your exact function without changes ---
semantic_report = get_semantic_drift_report(
    piv_df_translated,
    int_models_dict,
    threshold_std=0.8
)

# --- 4. Final Polish ---
if not semantic_report.empty:
    semantic_report = semantic_report.sort_values(by=['Period', 'Similarity'])
    print(f"Success! Detected {len(semantic_report)} drifting words.")
    display(semantic_report.head(10))
else:
    print("No drifts detected. Try lowering threshold_std to 0.5.")

In [ ]:
semantic_report.describe()

In [ ]:
semantic_report.to_csv('/content/drive/MyDrive/LSD-Analysis/Data/fasttext_neighbors_orthogonal_procrustes_NN.csv', index=False, encoding='utf-8-sig')

In [1]:
import pandas as pd
import numpy as np

In [2]:
semantic_report = pd.read_csv('../Data/fasttext_neighbors_orthogonal_procrustes_NN.csv', encoding='utf-8-sig')

In [3]:
semantic_report.head()

,Word,Period,Similarity,Neighbors_13,Neighbors_14,Neighbors_15,Neighbors_18,Neighbors_19,Neighbors_20
0,නම්,13_14,0.5977,"ඔහුගෙන්, පිණිසැ, එකී, ගියත්, විසින්, පිරුවත්, ...","ඇති, ටික, පසඟ, මූල, දන්, වටනා, අන්ය, සාර, උත්ත...",NaN,NaN,NaN,NaN
1,ඇත්තේ,13_14,0.5999,"විශෙෂයෙන්, සම්බන්ධයෙන්, සමූහයෙන්, මාර්ගයෙන්, භ...","වූයේය, ඔවුන්ගේ, ඇති, දෙදෙන, බුද්ධ, හෙවත්, උන්ව...",NaN,NaN,NaN,NaN
2,ඇති,13_14,0.6434,"ඇත්තේය, පිහිටියා, පැවැත්තේ, පිහිටුවන, තිබියදී,...","නම්, ඇත්තේ, හෙවත්, මූල, ටික, ගන්, වටනා, උත්ත, ...",NaN,NaN,NaN,NaN
3,උපන්,13_14,0.7021,"වණ්ණනා, ආයුෂ, ගොඩ, දිග, සාර, උස්, තුබුණු, අබි,...","අවුරුද්, වසන්නා, දෙව, වීර්ය, පිළිගෙණ, සත්වන, ව...",NaN,NaN,NaN,NaN
4,දක්වා,13_14,0.7276,"දක්වන්ට, ක්ෂණයෙහි, කරන්න, කරන්නට, පිරියම, භූමි...","පිළිබඳ, ඔවුන්ගේ, කරමින්, මෙන්, පිළිගෙන, නැඟී, ...",NaN,NaN,NaN,NaN


In [4]:
sorted_report = semantic_report.sort_values(by='Similarity', ascending=True)

In [5]:
sorted_report.head(6)

,Word,Period,Similarity,Neighbors_13,Neighbors_14,Neighbors_15,Neighbors_18,Neighbors_19,Neighbors_20
232,මෙන්,18_20,0.2527,NaN,NaN,NaN,"පුරණ, නැ, සමස්ත, නුගරුක, සංහිතා, හනු, හැඬීම්, ...",NaN,"දෝෂයෙන්, ක්රමයෙන්, වසයෙන්, නැතියෙන්, විශෙෂයෙන්..."
212,මෙන්,18_19,0.3181,NaN,NaN,NaN,"පුරණ, නැ, සමස්ත, නුගරුක, සංහිතා, හනු, හැඬීම්, ...","නිරන්තරයෙන්, තීරයෙහි, පදයෙන්, ඥානයෙන්, ක්රමයෙන...",NaN
35,මෙන්,13_18,0.4667,"වැදීමෙන්, අවධාරණයෙන්, සම්බන්ධයෙන්, ඥානයෙන්, කර...",NaN,NaN,"පුරණ, නැ, සමස්ත, නුගරුක, සංහිතා, හනු, හැඬීම්, ...",NaN,NaN
233,බැවින්,18_20,0.5203,NaN,NaN,NaN,"කෙළේ, පාස, හේතුවෙන්, වීමය, බවයි, වීමෙන්, වන්නේ...",NaN,"බැතියෙන්, දෙමින්, තිබෙන්නේ, කරණයෙහි, මින්, සම්..."
213,බැවින්,18_19,0.5357,NaN,NaN,NaN,"කෙළේ, පාස, හේතුවෙන්, වීමය, බවයි, වීමෙන්, වන්නේ...","හෙයින්, කියන්නා, පදාර්ථයෙහි, අන්තයෙහි, නැතිවන්...",NaN
157,මෙන්,15_18,0.5374,NaN,NaN,"පහසින්, කෙසේ, කුමක්, මිනිස්, එකල්හි, ස්ථානයන්හ...","පුරණ, නැ, සමස්ත, නුගරුක, සංහිතා, හනු, හැඬීම්, ...",NaN,NaN


In [6]:
semantic_report['Similarity'] = semantic_report['Similarity'].round(4)
split_periods = semantic_report['Period'].str.split('_', expand=True)
ref_num = split_periods[0]
target_num = split_periods[1]

semantic_report['Reference Century'] = ref_num + 'th'
semantic_report['Target Century'] = target_num + 'th'

In [7]:
def get_top_5_from_col(row, century_num):
    col_name = f'Neighbors_{century_num}'
    
    # Check if the column exists and has data for this row
    if col_name in row.index and pd.notna(row[col_name]) and row[col_name] != '':
        # Split by comma, clean up spaces, and grab the first 5 words
        words = [w.strip() for w in str(row[col_name]).split(',') if w.strip()]
        return ', '.join(words[:5])
    
    return '' # Return empty if no data found

# 4. Extract neighbors dynamically for both Reference and Target
semantic_report['Ref Century Neighbors'] = semantic_report.apply(
    lambda r: get_top_5_from_col(r, r['Period'].split('_')[0]), axis=1
)

semantic_report['Target Century Neighbors'] = semantic_report.apply(
    lambda r: get_top_5_from_col(r, r['Period'].split('_')[1]), axis=1
)


In [8]:
final_df = semantic_report[[
    'Word', 
    'Similarity',
    'Reference Century', 
    'Ref Century Neighbors', 
    'Target Century', 
    'Target Century Neighbors'
]].copy()

In [9]:
final_df.head()

,Word,Similarity,Reference Century,Ref Century Neighbors,Target Century,Target Century Neighbors
0,නම්,0.5977,13th,"ඔහුගෙන්, පිණිසැ, එකී, ගියත්, විසින්",14th,"ඇති, ටික, පසඟ, මූල, දන්"
1,ඇත්තේ,0.5999,13th,"විශෙෂයෙන්, සම්බන්ධයෙන්, සමූහයෙන්, මාර්ගයෙන්, භ...",14th,"වූයේය, ඔවුන්ගේ, ඇති, දෙදෙන, බුද්ධ"
2,ඇති,0.6434,13th,"ඇත්තේය, පිහිටියා, පැවැත්තේ, පිහිටුවන, තිබියදී",14th,"නම්, ඇත්තේ, හෙවත්, මූල, ටික"
3,උපන්,0.7021,13th,"වණ්ණනා, ආයුෂ, ගොඩ, දිග, සාර",14th,"අවුරුද්, වසන්නා, දෙව, වීර්ය, පිළිගෙණ"
4,දක්වා,0.7276,13th,"දක්වන්ට, ක්ෂණයෙහි, කරන්න, කරන්නට, පිරියම",14th,"පිළිබඳ, ඔවුන්ගේ, කරමින්, මෙන්, පිළිගෙන"


In [10]:
final_df = final_df.sort_values(by='Similarity', ascending=True)

In [11]:
final_df.head()

,Word,Similarity,Reference Century,Ref Century Neighbors,Target Century,Target Century Neighbors
232,මෙන්,0.2527,18th,"පුරණ, නැ, සමස්ත, නුගරුක, සංහිතා",20th,"දෝෂයෙන්, ක්රමයෙන්, වසයෙන්, නැතියෙන්, විශෙෂයෙන්"
212,මෙන්,0.3181,18th,"පුරණ, නැ, සමස්ත, නුගරුක, සංහිතා",19th,"නිරන්තරයෙන්, තීරයෙහි, පදයෙන්, ඥානයෙන්, ක්රමයෙන්"
35,මෙන්,0.4667,13th,"වැදීමෙන්, අවධාරණයෙන්, සම්බන්ධයෙන්, ඥානයෙන්, කර...",18th,"පුරණ, නැ, සමස්ත, නුගරුක, සංහිතා"
233,බැවින්,0.5203,18th,"කෙළේ, පාස, හේතුවෙන්, වීමය, බවයි",20th,"බැතියෙන්, දෙමින්, තිබෙන්නේ, කරණයෙහි, මින්"
213,බැවින්,0.5357,18th,"කෙළේ, පාස, හේතුවෙන්, වීමය, බවයි",19th,"හෙයින්, කියන්නා, පදාර්ථයෙහි, අන්තයෙහි, නැතිවන්නේය"


In [12]:
final_df['Ref Century Neighbors'] = '\\iffalse ' + final_df['Ref Century Neighbors'].astype(str) + ' \\fi'
final_df['Target Century Neighbors'] = '\\iffalse ' + final_df['Target Century Neighbors'].astype(str) + ' \\fi'

In [13]:
float_formatter = lambda x: f"{x:.4f}".rstrip('0').rstrip('.') if isinstance(x, float) else str(x)

In [14]:
final_df.head(5).to_latex('../Data/FastText_Semantic_Drift_Report_NPA.tex', index=False, encoding='utf-8-sig', escape=False, formatters={'Similarity': float_formatter})

In [15]:
final_df.to_csv('../Data/FastTex_dip_neighbors_similarity_matrix_NPA_final.csv', index=False, encoding='utf-8-sig')    

In [16]:
selected_words = ['රැස්', 'රාජ', 'සංඛ්යාත']
filtered_df = final_df[final_df["Word"].isin(selected_words)]

In [17]:
filtered_df

,Word,Similarity,Reference Century,Ref Century Neighbors,Target Century,Target Century Neighbors
223,රාජ,0.6980,18th,"\iffalse සර්වාකාරයෙන්, ස්වරයෙන්, නිරායාසයෙන්, ...",19th,"\iffalse මංගල, ගරුඩ, අතීත, දත්, වර්ෂ \fi"
180,රැස්,0.7044,15th,"\iffalse විනාස, ආයු, රහස්ය, රත්න, බුදුරැස් \fi",19th,"\iffalse ස්පර්ශයෙන්, භාෂාවෙන්, රත්නයෙන්, යක්ෂය..."
83,රැස්,0.7112,14th,"\iffalse රැස්කරවා, පිළිගෙණ, පර්යංකයෙන්, මධ්යයෙ...",15th,"\iffalse විනාස, ආයු, රහස්ය, රත්න, බුදුරැස් \fi"
54,සංඛ්යාත,0.7178,13th,"\iffalse ආකාශ, බොධිය, අමාත්ය, විශේෂ, කුලෙහි \fi",19th,"\iffalse සංඛ්යාවක්, ක්රමයෙන්, ශරීරයෙන්, ප්රමාණ..."
16,රැස්,0.7219,13th,"\iffalse ඇත්තා, කරන්නේ, පිරියම, පිළිකෙව්, ස්ථා...",15th,"\iffalse විනාස, ආයු, රහස්ය, රත්න, බුදුරැස් \fi"
205,රැස්,0.7298,15th,"\iffalse විනාස, ආයු, රහස්ය, රත්න, බුදුරැස් \fi",20th,"\iffalse අනුදැනීම, විස්සක්, ආරක්ෂා, සමාදානයන්,..."
256,රාජ,0.7335,19th,"\iffalse මංගල, ගරුඩ, අතීත, දත්, වර්ෂ \fi",20th,"\iffalse පුරාතන, මුඳුන්, ග්රන්ථ, පෙළපතක, නිවටය..."
128,රාජ,0.7410,14th,"\iffalse කවුරුන්ගේ, සුදුසු, කල්පයට, පණ්ඩිත, පු...",19th,"\iffalse මංගල, ගරුඩ, අතීත, දත්, වර්ෂ \fi"
153,රැස්,0.7675,14th,"\iffalse රැස්කරවා, පිළිගෙණ, පර්යංකයෙන්, මධ්යයෙ...",20th,"\iffalse අනුදැනීම, විස්සක්, ආරක්ෂා, සමාදානයන්,..."
24,සංඛ්යාත,0.7793,13th,"\iffalse ආකාශ, බොධිය, අමාත්ය, විශේෂ, කුලෙහි \fi",15th,"\iffalse කර්මාන්තයෙහි, කාරණයෙන්, භාවයෙන්, ආකාර..."


In [18]:
# 1. Condition for 'රැස්' (14th -> 15th Century)
cond_ras = (
    (final_df["Word"] == "රැස්") 
    & (final_df["Reference Century"] == "14th") 
    & (final_df["Target Century"] == "15th")
)

# 2. Condition for 'රාජ' (19th -> 20th Century)
cond_raja = (
    (final_df["Word"] == "රාජ") 
    & (final_df["Reference Century"] == "19th") 
    & (final_df["Target Century"] == "20th")
)

# 3. Condition for 'සංඛ්යාත' (13th -> 19th Century)
cond_samyak = (
    (final_df["Word"] == "සංඛ්යාත") 
    & (final_df["Reference Century"] == "13th") 
    & (final_df["Target Century"] == "19th")
)

# Filter the NPA dataset using the logical OR operator (|)
filtered_df = final_df[cond_ras | cond_raja | cond_samyak]

In [19]:
filtered_df = filtered_df.sort_values(by='Similarity', ascending=True)

In [20]:
filtered_df

,Word,Similarity,Reference Century,Ref Century Neighbors,Target Century,Target Century Neighbors
83,රැස්,0.7112,14th,"\iffalse රැස්කරවා, පිළිගෙණ, පර්යංකයෙන්, මධ්යයෙ...",15th,"\iffalse විනාස, ආයු, රහස්ය, රත්න, බුදුරැස් \fi"
54,සංඛ්යාත,0.7178,13th,"\iffalse ආකාශ, බොධිය, අමාත්ය, විශේෂ, කුලෙහි \fi",19th,"\iffalse සංඛ්යාවක්, ක්රමයෙන්, ශරීරයෙන්, ප්රමාණ..."
256,රාජ,0.7335,19th,"\iffalse මංගල, ගරුඩ, අතීත, දත්, වර්ෂ \fi",20th,"\iffalse පුරාතන, මුඳුන්, ග්රන්ථ, පෙළපතක, නිවටය..."


In [21]:
filtered_df.to_latex('../Data/FastText_Semantic_Drift_Report_NPA.tex', index=False, encoding='utf-8-sig', escape=False, formatters={'Similarity': float_formatter})